### Initial attempt by D. Barco on Fri 03-Oct-2025 @ 15h55
#### - attended last few minutes of Friday tutorial w/ Edward
#### - re-attempting on Tue 07-Oct-2025 @ 14h25 - 23h45 (sporadic / intermittent progress throughout the afternoon & evening)
#### - final attempt on Thu 09-Oct-2025 @ 13h30 (will submit whatever I can complete)
#### - completed (to the best of my ability) on Thu 09-Oct-2025 @ 16h25

# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [1]:
# Load the libraries as required.
# Load environment variables ("magic" jupyter notebook commands)
%load_ext dotenv
%dotenv 

# Add src to path
import os
import sys
sys.path.append(os.getenv('SRC_DIR'))

# Standard libraries
import random
import pandas as pd
import numpy as np

# set random state = 42
random.seed(42)

In [2]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


In [3]:
fires_dt # <-- IGNORE (just checking)

,coord_x,coord_y,month,day,ffmc,dmc,dc,isi,temp,rh,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.00
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.00
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.00
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.00
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,4,3,aug,sun,81.6,56.7,665.6,1.9,27.8,32,2.7,0.0,6.44
513,2,4,aug,sun,81.6,56.7,665.6,1.9,21.9,71,5.8,0.0,54.29
514,7,4,aug,sun,81.6,56.7,665.6,1.9,21.2,70,6.7,0.0,11.16
515,1,4,aug,sat,94.4,146.0,614.7,11.3,25.6,42,4.0,0.0,0.00


# Get X and Y

Create the features data frame and target data.

In [4]:
# features data frame... Fri 03-Oct-2025 @ 16h55
# continuing Tue 07-Oct-2025 @ 14h25
X = fires_dt.drop(columns = ['area'])
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 12 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
dtypes: float64(7), int64(3), object(2)
memory usage: 48.6+ KB


In [5]:
# target data... Fri 03-Oct-2025 @ 16h55
# continuing Tue 07-Oct-2025 @ 14h25
y = fires_dt['area']
y.info()

<class 'pandas.core.series.Series'>
RangeIndex: 517 entries, 0 to 516
Series name: area
Non-Null Count  Dtype  
--------------  -----  
517 non-null    float64
dtypes: float64(1)
memory usage: 4.2 KB


In [6]:
y.describe() # <-- IGNORE (just checking)

count     517.000000
mean       12.847292
std        63.655818
min         0.000000
25%         0.000000
50%         0.520000
75%         6.570000
max      1090.840000
Name: area, dtype: float64

In [7]:
# Create train and test splits
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [8]:
# import needed libraries
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline

In [9]:
preproc1 = ColumnTransformer(
    transformers = [
        ('numeric_transformer', StandardScaler(), ['coord_x', 'coord_y',
                                                   'ffmc', 'dmc', 'dc', 'isi',
                                                   'temp', 'rh', 'wind', 'rain']),
        ('onehot', OneHotEncoder(handle_unknown = 'infrequent_if_exist'), ['month', 'day']),
         ], remainder = 'passthrough'
)

preproc1 # <-- IGNORE (just checking)

ColumnTransformer(remainder='passthrough',
                  transformers=[('numeric_transformer', StandardScaler(),
                                 ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc',
                                  'isi', 'temp', 'rh', 'wind', 'rain']),
                                ('onehot',
                                 OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                 ['month', 'day'])])

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [10]:
preproc2 = ColumnTransformer(
    transformers = [
        ('numeric_transformer', PowerTransformer(), ['coord_x', 'coord_y',
                                                   'ffmc', 'dmc', 'dc', 'isi',
                                                   'temp', 'rh', 'wind', 'rain']),
        ('onehot', OneHotEncoder(handle_unknown = 'infrequent_if_exist'), ['month', 'day']),
         ], remainder = 'passthrough'
)

preproc2 # <-- IGNORE (just checking)

ColumnTransformer(remainder='passthrough',
                  transformers=[('numeric_transformer', PowerTransformer(),
                                 ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc',
                                  'isi', 'temp', 'rh', 'wind', 'rain']),
                                ('onehot',
                                 OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                 ['month', 'day'])])

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [11]:
# Pipeline A = preproc1 + baseline
Pipeline_A = Pipeline(
    [
        ('preprocessing', preproc1),
        ('regressor', KNeighborsRegressor(n_neighbors = 5))
    ]
)

Pipeline_A # <-- IGNORE (just checking)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transformer',
                                                  StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  ['month', 'day'])])),
                ('regressor', KNeighborsRegressor())])

In [12]:
# Pipeline B = preproc2 + baseline
Pipeline_B = Pipeline(
    [
        ('preprocessing', preproc2),
        ('regressor', Lasso(alpha = 1.0))
    ]
)

Pipeline_B # <-- IGNORE (just checking)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numeric_transformer',
                                                  PowerTransformer(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  ['month', 'day'])])),
                ('regressor', Lasso())])

In [13]:
# Pipeline C = preproc1 + advanced model
Pipeline_C = Pipeline(
    [
        ('preprocessing', preproc1),
        ('regressor', ??() ) # I'm not quite sure how to do the advanced (Ensemble) model :-(
    ]
)

Pipeline_C # <-- IGNORE (just checking)

SyntaxError: invalid syntax (2297484720.py, line 5)

In [14]:
# Pipeline D = preproc2 + advanced model
Pipeline_D = Pipeline(
    [
        ('preprocessing', preproc2),
        ('regressor', ??() ) # I'm not quite sure how to do the advanced (Ensemble) model :-(
    ]
)

Pipeline_D # <-- IGNORE (just checking)

SyntaxError: invalid syntax (2651683869.py, line 5)

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

## Pipeline A

In [15]:
from sklearn.model_selection import GridSearchCV
grid_cv_A = GridSearchCV(
    estimator = Pipeline_A,
    param_grid = {'regressor__n_neighbors': [3, 5, 7, 9]},
    scoring = 'neg_max_error',
    cv = 5,
    refit = "neg_log_loss")

In [16]:
grid_cv_A.fit(X_train, y_train) # Train (fit) the model

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('numeric_transformer',
                                                                         StandardScaler(),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('onehot',
                                                                         OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                                         ['month',
                                                                          'day'])])),
                                       ('regressor', KNeighborsRegressor())]),
             param_grid={'regressor__n_neighbors': [3, 5, 7, 9]},
             refit='neg_log_loss', scoring='neg_max_error')

In [17]:
Y_pred_train_A = grid_cv_A.predict(X_train) # Predict on training data

Y_pred_test_A = grid_cv_A.predict(X_test) # Predict on test data

## Pipeline B

In [18]:
grid_cv_B = GridSearchCV(
    estimator = Pipeline_B,
    param_grid = {'regressor__alpha': np.arange(0.01, 1.00, 0.01)},
    scoring = 'neg_max_error',
    cv = 5,
    refit = "neg_log_loss")

In [19]:
grid_cv_B.fit(X_train, y_train) # Train (fit) the model

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('numeric_transformer',
                                                                         PowerTransformer(),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('onehot',
                                                                         OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                                         ['month',
                                                                          'day'])])),
                                       ('regressor', Lasso())]),
             param_grid={...
       0.34, 0.35, 0.36, 0.37, 0.38, 0.39, 0.4 , 0.41, 0.42, 0.43, 0.44,
       0.45, 0.46, 0.47, 0.48, 0.49, 0.5 , 0.51, 0.52, 0.53, 0.54, 0.55,
       0.56, 0.57, 0.58, 0.59, 0.6 , 0.61, 0.62, 0.63, 0.64, 0.65, 0.66,
       0.67, 0.68, 0.69, 0.7 , 0.71, 0.72, 0.73, 0.74, 0.75, 0.76, 0.77,
       0.78, 0.79, 0.8 , 0.81, 0.82, 0.83, 0.84, 0.85, 0.86, 0.87, 0.88,
       0.89, 0.9 , 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99])},
             refit='neg_log_loss', scoring='neg_max_error')

In [20]:
Y_pred_train_B = grid_cv_B.predict(X_train) # Predict on training data

Y_pred_test_B = grid_cv_B.predict(X_test) # Predict on test data

## Pipeline C

In [ ]:
# incomplete (not sure how to do the advanced model)

## Pipeline D

In [ ]:
# incomplete (not sure how to do the advanced model)

# Evaluate

+ Which model has the best performance?

> Comparing my baseline models:
>Pipeline A = StandardScaler + KNeighborsRegressor
>Pipeline B = PowerTransformer + Lasso
>
>Pipeline A model took 2.4 s to train / fit
>
>Pipeline B model took 26.1 s to train / fit (So it is 10x slower than Pipeline A)
>
>
>The results for the error metrics that I selected are identical.
>
>Therefore, **I would likely go with the simple (baseline) model (Pipeline A)**

## Results Pipeline A

In [21]:
# generate some metrics for comparison
from sklearn.metrics import mean_absolute_error, max_error
results_A = {
    'max_error_train': max_error(y_train, Y_pred_train_A),
    'max_error_test': max_error(y_test, Y_pred_test_A),
    'mean_absolute_error_train': mean_absolute_error(y_train, Y_pred_train_A),
    'mean_absolute_error_test': mean_absolute_error(y_test, Y_pred_test_A)  
}
results_A

{'max_error_train': np.float64(661.7277777777778),
 'max_error_test': np.float64(1072.5855555555554),
 'mean_absolute_error_train': 14.971068065644337,
 'mean_absolute_error_test': 24.368237179487174}

## Results Pipeline B

In [22]:
# generate some metrics for comparison
from sklearn.metrics import mean_absolute_error, max_error
results_B = {
    'max_error_train': max_error(y_train, Y_pred_train_A),
    'max_error_test': max_error(y_test, Y_pred_test_A),
    'mean_absolute_error_train': mean_absolute_error(y_train, Y_pred_train_A),
    'mean_absolute_error_test': mean_absolute_error(y_test, Y_pred_test_A)  
}
results_B

{'max_error_train': np.float64(661.7277777777778),
 'max_error_test': np.float64(1072.5855555555554),
 'mean_absolute_error_train': 14.971068065644337,
 'mean_absolute_error_test': 24.368237179487174}

# Export

+ Save the best performing model to a pickle file.

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

*(Answer here.)*

> In predicting Forest Fire Area (burned), I would likely remove any features that are not _directly_ weather related. For example, coord_X, coord_Y, month, day and the other indices FFMC, DMC, DC & ISI.

> I think Temperature, Humidity (RH), wind and rain would be the four best predictors of fire and thus (burned) area affected.

> Diogo Brian Barco (GH:dibarco7) on Thu 09-Oct-2025 @ 16h25

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.